# 03 · Message Passing Framework

## What this notebook covers
All modern GNNs share a common structure: message passing. This notebook implements the message passing framework from scratch to build intuition before using PyTorch Geometric.

## The unified message passing view
Every GNN layer does three operations:

```
1. MESSAGE:   m_ij = φ(h_i, h_j, e_ij)      — compute messages from each neighbour
2. AGGREGATE: m_i  = ⊕_{j ∈ N(i)} m_ij      — aggregate messages (sum / mean / max)
3. UPDATE:    h_i' = γ(h_i, m_i)             — update node representation
```

Where:
- h_i = current node i embedding
- φ = message function (learnable MLP, or just the neighbour embedding)
- ⊕ = aggregation (sum, mean, max — each has different expressiveness)
- γ = update function (GRU or MLP)

The choice of message, aggregation, and update functions defines the GNN variant:
- **GCN**: φ = linear transform, ⊕ = normalised sum, γ = identity
- **GraphSAGE**: ⊕ = mean/max/LSTM, γ = linear
- **GAT**: messages are weighted by learned attention scores
- **GIN**: most expressive; key for graph classification

In [ ]:
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(42); np.random.seed(42)

# ── Build a toy graph ──────────────────────────────────────────────────────────
G = nx.karate_club_graph()
n_nodes = G.number_of_nodes()

# Adjacency (with self-loops) and degree matrix
A = nx.to_numpy_array(G) + np.eye(n_nodes)  # add self-loops
D = np.diag(A.sum(axis=1))
D_inv_sqrt = np.diag(1 / np.sqrt(A.sum(axis=1)))
A_hat = D_inv_sqrt @ A @ D_inv_sqrt  # normalised adjacency

# Node features: one-hot degree encoding (simple baseline)
degrees = np.array([G.degree(n) for n in G.nodes()])
X = np.eye(n_nodes)  # identity features (most expressive per Xu et al. 2019)
X_t = torch.FloatTensor(X)
A_t = torch.FloatTensor(A_hat)

labels = {n: 0 if G.nodes[n]["club"] == "Mr. Hi" else 1 for n in G.nodes()}
y = torch.LongTensor([labels[n] for n in G.nodes()])
print(f"Graph: {n_nodes} nodes  |  Feature dim: {X.shape[1]}  |  Classes: 2")

In [ ]:
# ── Message passing from scratch ───────────────────────────────────────────────
class MessagePassingLayer(nn.Module):
    """
    Generic message passing:
      h_i' = UPDATE(h_i, AGGREGATE({MESSAGE(h_i, h_j) : j in N(i)}))
    This implements GCN-style normalised sum aggregation.
    """
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))

    def forward(self, X, A_norm):
        # MESSAGE + AGGREGATE: A_norm @ X computes normalised sum of neighbour features
        agg = A_norm @ X          # (N, in_dim)
        # UPDATE: linear transform + bias
        return F.relu(self.W(agg) + self.bias)

class SimpleGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.conv1 = MessagePassingLayer(in_dim, hidden_dim)
        self.conv2 = MessagePassingLayer(hidden_dim, out_dim)

    def forward(self, X, A):
        h = self.conv1(X, A)
        h = F.dropout(h, p=0.3, training=self.training)
        return self.conv2(h, A)

model = SimpleGCN(in_dim=n_nodes, hidden_dim=16, out_dim=2)
opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

# Train (semi-supervised: use 50% of nodes as training)
train_mask = torch.zeros(n_nodes, dtype=torch.bool)
train_mask[:n_nodes//2] = True

losses, accs = [], []
for epoch in range(200):
    model.train()
    out = model(X_t, A_t)
    loss = F.cross_entropy(out[train_mask], y[train_mask])
    opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        acc = (out.argmax(dim=1) == y).float().mean().item()
    losses.append(loss.item()); accs.append(acc)

print(f"Final loss: {losses[-1]:.4f}  |  Final accuracy (all nodes): {accs[-1]:.3f}")

In [ ]:
# Visualise learned embeddings
model.eval()
with torch.no_grad():
    # Extract hidden representations from conv1
    h1 = model.conv1(X_t, A_t).numpy()

from sklearn.manifold import TSNE
h1_2d = TSNE(n_components=2, random_state=0, perplexity=5).fit_transform(h1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colours = ["steelblue" if yi == 0 else "tomato" for yi in y.numpy()]
axes[0].scatter(h1_2d[:, 0], h1_2d[:, 1], c=colours, s=80, alpha=0.8)
for i in range(n_nodes):
    axes[0].annotate(str(i), h1_2d[i], fontsize=6, alpha=0.5)
axes[0].set_title("GCN hidden representations (t-SNE)")

axes[1].plot(losses, color="steelblue", label="Loss")
ax2 = axes[1].twinx()
ax2.plot(accs, color="tomato", label="Accuracy")
axes[1].set_xlabel("Epoch"); axes[1].set_title("Training curve")
axes[1].legend(loc="upper left"); ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig(f"{base}/03_message_passing.png", dpi=120)
plt.show()

## Aggregation comparison
| Aggregator | Expressiveness | Notes |
|-----------|---------------|-------|
| **Sum** | Highest (counts neighbours) | Distinguishes different-size neighbourhoods |
| **Mean** | Medium (averages structure) | Invariant to neighbourhood size |
| **Max** | Lowest (one representative) | Good for detecting any presence of a feature |

GIN (Graph Isomorphism Network, Xu et al. 2019) proved that sum aggregation is theoretically optimal — as powerful as the Weisfeiler-Lehman graph isomorphism test. Most modern architectures use sum or attention-weighted aggregation.